# Classical Model Training OvA Using the CONNIE Dataset (with img descriptor)

In [2]:
%run ./../notebook_init.py

import os
import uproot
import optuna
import mlflow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from glob import glob
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             confusion_matrix,
                             precision_score,
                             recall_score,
                             f1_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from skimage.morphology import skeletonize
from scipy.ndimage import convolve
from skimage.measure import find_contours

from core import DATA_FOLDER, RESULTS_FOLDER
from scripts.connie_training_utils import (Seed, extract_skeleton_features,
                                           extract_fourier_descriptors)

In [3]:
train_data = os.path.join(DATA_FOLDER, "train_data_root_full")

In [4]:
seed = Seed()

In [5]:
#categories = ["Alpha", "Blob", "Diffusion Hit", "Electron", "Muon", "Others"]
categories = ["Blob", "Diffusion Hit", "Electron", "Muon", "Others"]

branch_name = "hitSumm"

In [6]:
all_data_list = []
all_data_list_excluded_vars = []

print("Starting data loading")
for category in categories:
    category_path = os.path.join(train_data, category)
    root_files = glob(os.path.join(category_path, "*.root"))

    if not root_files:
        print(f"Warning: No .root files found in {category_path}")
        continue

    print(f"Processing category: {category} ({len(root_files)} files)")
    for idx, file_path in enumerate(root_files):
        try:
            with uproot.open(file_path) as file:
                if branch_name not in file:
                    print(f"Warning: TTree '{branch_name}' not found in {file_path}. Skipping.")
                    continue
                file_branch = file[branch_name]
                df = file_branch.arrays(library="pd")
                df['label'] = category
                all_data_list.append(df)

        except Exception as e:
            print(f"Error processing file {file_path}: {e}")


Starting data loading
Processing category: Blob (351 files)
Processing category: Diffusion Hit (44 files)
Processing category: Electron (366 files)
Processing category: Muon (2596 files)
Processing category: Others (220 files)


Combine all DataFrames into a single DataFrame

In [7]:
if all_data_list:
    df_combined = pd.concat(all_data_list, ignore_index=True)
    print(f"Successfully loaded {len(df_combined)} rows of data.")
else:
    print("No data loaded.")

Successfully loaded 3577 rows of data.


* Calculate mean of ePix and level to be used as features

In [8]:
df_processed = df_combined.copy()

df_processed["ePixMean"] = df_processed["ePix"].apply(np.mean)
df_processed["levelMean"] = df_processed["level"].apply(np.mean)

In [9]:
skeleton_features = df_processed.apply(
    lambda row: extract_skeleton_features(row['xPix'],
                                          row['yPix']),
    axis=1)

fd_features = df_processed.apply(
    lambda row: extract_fourier_descriptors(row['xPix'],
                                            row['yPix'], 10),
    axis=1)


new_skeleton_features_df = pd.json_normalize(skeleton_features)
new_fd_features_df = pd.json_normalize(fd_features)

df_processed = pd.concat([df_processed.reset_index(drop=True),
                          new_skeleton_features_df], axis=1)
df_processed = pd.concat([df_processed.reset_index(drop=True),
                          new_fd_features_df], axis=1)


* Remove features with more than one dimension, such as xPix and yPix
* Remove "flag", as we already filtered for only valid events
* Drop columns with no variance

In [10]:
df_processed = df_processed.drop(columns=["label", "xPix", "yPix", "level", "ePix", "flag"])

# Drop columns with no variance
df_processed = df_processed.loc[:, df_processed.nunique() > 1]

In [11]:
print(df_processed.columns)

Index(['runID', 'imgID', 'chid', 'ohdu', 'skpID', 'expoStart', 'Gain', 'imgG',
       'Noise', 'SER', 'ExpTime', 'NpixAC', 'DeltaT', 'xMin', 'xMax', 'yMin',
       'yMax', 'E0', 'n0', 'xBary0', 'yBary0', 'xVar0', 'yVar0', 'E1', 'n1',
       'xBary1', 'yBary1', 'xVar1', 'yVar1', 'nSavedPix', 'nxPix', 'nyPix',
       'nlevel', 'nePix', 'EventID', 'ePixMean', 'levelMean',
       'skeleton_length', 'num_branches', 'num_endpoints',
       'branch_to_end_ratio', 'skeleton_area_ratio', 'skeleton_tortuosity',
       'skeleton_linearity_deviation', 'skeleton_curvature_sum', 'FD_2',
       'FD_3', 'FD_4', 'FD_5', 'FD_6', 'FD_7', 'FD_8', 'FD_9', 'FD_10',
       'FD_11'],
      dtype='object')


In [12]:
print(df_processed)

      runID  imgID  chid  ohdu  skpID   expoStart        Gain        imgG  \
0       118   1786    14   114      1  1660886247  438.962463  436.332947   
1       118   1789    14   114      1  1660894898  438.962463  432.576416   
2       118   1819    14   114      1  1660981329  438.962463  439.316559   
3       118    355    14   114      1  1656762273  438.962463  432.167206   
4       118    391    14   114      1  1656866649  438.962463  435.928986   
...     ...    ...   ...   ...    ...         ...         ...         ...   
3572    125    318    14   114      1  1667160285  439.492920  435.454010   
3573    125    322    14   114      1  1667197901  439.492920  434.920044   
3574    125    322    14   114      1  1667197901  439.492920  434.920044   
3575    125    325    14   114      1  1667226120  439.492920  442.380890   
3576    125    326    14   114      1  1667235530  439.492920  435.004517   

         Noise       SER  ...      FD_2      FD_3      FD_4      FD_5  \
0 

Calculating the correlation between features

In [13]:
corr_df_combined = df_processed.corr()
corr_pairs = corr_df_combined.unstack()
# Filter out self-correlations
filtered = corr_pairs[corr_pairs != 1.0]
# Remove duplicate mirror entries
filtered = filtered.drop_duplicates()
# Find correlations above 0.9
high_corr = filtered[filtered.abs() > 0.9]
print(high_corr.sort_values(ascending=False))

ohdu          skpID                  1.000000
yBary0        yBary1                 1.000000
xBary0        xBary1                 1.000000
yVar0         yVar1                  1.000000
xVar0         xVar1                  1.000000
E0            E1                     0.999995
yMax          yBary0                 0.999305
              yBary1                 0.999305
yMin          yBary1                 0.999300
              yBary0                 0.999300
n0            n1                     0.998214
yMin          yMax                   0.997349
xMax          xBary1                 0.995830
              xBary0                 0.995830
xMin          xBary0                 0.995590
              xBary1                 0.995590
              xMax                   0.983850
skpID         Gain                   0.976886
ohdu          Gain                   0.976886
n1            skeleton_length        0.970822
num_branches  branch_to_end_ratio    0.966292
n0            skeleton_length     

Removing features from the dataframe

In [14]:
drop_cols = [
    # ===== DETECTOR METADATA (Not event-specific) =====
    "ohdu",        # Duplicate of skpID (correlation = 1.0 with skpID)
    "chid",        # Duplicate of skpID (correlation = -1.0)
    "skpID",       # Sensor ID - same for all events from same sensor
    "runID",       # Run ID - same for all events in same run
    "imgID",       # Image ID - same for all events in same image
    "Gain",        # Global gain - same for all events in same run
    
    # ===== IMAGE-LEVEL METADATA (Same for all events in image) =====
    "expoStart",   # Exposure start timestamp
    "DeltaT",      # Readout time
    "NpixAC",      # Number of unmasked active pixels
    
    # ===== EXACT DUPLICATES - Keep Level 0, Drop Level 1 =====
    "E1",          # correlation = 1.000 with E0
    "n1",          # correlation = 0.998 with n0
    "xBary1",      # correlation = 1.000 with xBary0
    "yBary1",      # correlation = 1.000 with yBary0
    "xVar1",       # correlation = 1.000 with xVar0
    "yVar1",       # correlation = 1.000 with yVar0
    
    # ===== REDUNDANT SIZE FEATURES (All ≈ n0, correlation ~0.998) =====
    "nSavedPix",   # Total saved pixels
    "nxPix",       # Length of xPix array
    "nyPix",       # Length of yPix array
    "nlevel",      # Number of levels
    "nePix",       # Number of energy pixels
    
    # ===== HIGH CORRELATION WITH BARYCENTER (>0.995) =====
    "xMin",        # correlation = 0.996 with xBary0
    "xMax",        # correlation = 0.996 with xBary0
    "yMin",        # correlation = 0.999 with yBary0
    "yMax",        # correlation = 0.999 with yBary0
    
    # ===== HIGH CORRELATION WITH OTHER FEATURES (>0.96) =====
    "skeleton_length",      # correlation = 0.963 with n0
    "branch_to_end_ratio",  # correlation = 0.966 with num_branches
]


df_processed_final = df_processed.drop(columns=drop_cols)


In [15]:
print(df_processed_final.columns)

Index(['imgG', 'Noise', 'SER', 'ExpTime', 'E0', 'n0', 'xBary0', 'yBary0',
       'xVar0', 'yVar0', 'EventID', 'ePixMean', 'levelMean', 'num_branches',
       'num_endpoints', 'skeleton_area_ratio', 'skeleton_tortuosity',
       'skeleton_linearity_deviation', 'skeleton_curvature_sum', 'FD_2',
       'FD_3', 'FD_4', 'FD_5', 'FD_6', 'FD_7', 'FD_8', 'FD_9', 'FD_10',
       'FD_11'],
      dtype='object')


In [17]:
df_processed_final

,imgG,Noise,SER,ExpTime,E0,n0,xBary0,yBary0,xVar0,yVar0,...,FD_2,FD_3,FD_4,FD_5,FD_6,FD_7,FD_8,FD_9,FD_10,FD_11
0,436.332947,0.144676,0.043237,0.040100,37882.824219,46.0,96.147751,565.805847,1.424701,0.784381,...,0.023160,0.018566,0.023334,0.012190,0.015219,0.004441,0.020538,0.007991,0.002897,0.009431
1,432.576416,0.144874,0.039609,0.040036,3895.175049,29.0,200.953415,741.643677,0.698164,0.685255,...,0.034351,0.003645,0.019168,0.019998,0.021997,0.005224,0.013913,0.006668,0.002529,0.012280
2,439.316559,0.150651,0.040646,0.040281,19051.687500,74.0,307.407104,484.713867,2.152307,1.517229,...,0.011675,0.010446,0.011962,0.009663,0.006360,0.016827,0.010316,0.010999,0.004776,0.011871
3,432.167206,0.159159,0.031582,0.040062,7002.749023,55.0,297.490082,452.348083,1.254075,1.308563,...,0.063668,0.007542,0.017306,0.017883,0.007307,0.001753,0.012148,0.012010,0.011421,0.008564
4,435.928986,0.168986,0.035983,0.040133,3531.689941,25.0,282.538635,426.804108,0.576674,0.561895,...,0.041159,0.021139,0.011654,0.006386,0.004919,0.008106,0.005359,0.005463,0.013176,0.007856
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3572,435.454010,0.159766,0.029164,0.038945,256159.718750,890.0,116.783394,571.474060,292.994507,1279.896851,...,0.161215,0.155339,0.046703,0.065233,0.023637,0.022012,0.040982,0.016902,0.012493,0.016188
3573,434.920044,0.156878,0.037900,0.039095,145683.859375,253.0,238.344269,410.463837,71.642189,24.733194,...,0.119398,0.183856,0.032468,0.011408,0.039258,0.007144,0.020211,0.004643,0.007778,0.014158
3574,434.920044,0.156878,0.037900,0.039095,162018.453125,453.0,137.598236,766.079163,33.260826,350.025543,...,0.218410,0.203693,0.056164,0.052254,0.037567,0.032895,0.012330,0.025503,0.003635,0.011054
3575,442.380890,0.154990,0.031160,0.039189,123933.203125,290.0,190.205551,251.990463,152.352386,6.299669,...,0.017396,0.135625,0.023894,0.070282,0.015168,0.022379,0.027621,0.001790,0.015769,0.003578


## Hyperparameters Tuning

* Use k-fold cross-validation for training and validation

In [15]:
from pathlib import Path
mlflow.set_tracking_uri(Path(DATA_FOLDER) / "mlruns")
# mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

x_train = df_processed_final.copy()
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(df_combined["label"])

for i, class_name in enumerate(label_encoder.classes_):
    print(f"Class ID {i}: {class_name}")

k_folds = 5
kf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seed.get_seed())

Class ID 0: Blob
Class ID 1: Diffusion Hit
Class ID 2: Electron
Class ID 3: Muon
Class ID 4: Others


In [16]:
def cross_val_score_with_logging(model, x_train_cv,
                                y_train_bin,
                                trial_number,
                                class_name,
                                is_xgboost=False):

    val_scores, train_scores = [], []

    val_f1_scores, val_precision_scores, val_recall_scores = [], [], []
    train_f1_scores, train_precision_scores, train_recall_scores = [], [], []

    val_f1_macro_scores, train_f1_macro_scores = [], []

    all_val_preds, all_val_true = [], []
    all_train_preds, all_train_true = [], []

    for train_idx, val_idx in kf.split(x_train_cv, y_train_bin):
        fold_model = clone(model)

        x_tr, x_val = x_train_cv.iloc[train_idx], x_train_cv.iloc[val_idx]
        y_tr, y_val = y_train_bin[train_idx], y_train_bin[val_idx]

        scaler = StandardScaler()
        x_tr_scaled = scaler.fit_transform(x_tr)
        x_val_scaled = scaler.transform(x_val)

        x_tr_scaled = pd.DataFrame(x_tr_scaled, columns=x_tr.columns, index=x_tr.index)
        x_val_scaled = pd.DataFrame(x_val_scaled, columns=x_val.columns, index=x_val.index)

        pos_count = np.sum(y_tr == 1)
        neg_count = np.sum(y_tr == 0)

        if pos_count == 0 or neg_count == 0:
            scale_pos_weight = 1.0
        else:
            scale_pos_weight = neg_count / pos_count

        if is_xgboost:
            fold_model.set_params(scale_pos_weight=scale_pos_weight)

        fold_model.fit(x_tr_scaled, y_tr)

        y_tr_pred = fold_model.predict(x_tr_scaled)
        y_val_pred = fold_model.predict(x_val_scaled)

        train_scores.append(accuracy_score(y_tr, y_tr_pred))
        val_scores.append(accuracy_score(y_val, y_val_pred))

        train_f1_scores.append(f1_score(y_tr, y_tr_pred, average="binary", zero_division=0))
        train_precision_scores.append(precision_score(y_tr, y_tr_pred, average="binary", zero_division=0))
        train_recall_scores.append(recall_score(y_tr, y_tr_pred, average="binary", zero_division=0))

        val_f1_scores.append(f1_score(y_val, y_val_pred, average="binary", zero_division=0))
        val_precision_scores.append(precision_score(y_val, y_val_pred, average="binary", zero_division=0))
        val_recall_scores.append(recall_score(y_val, y_val_pred, average="binary", zero_division=0))


        train_f1_macro_scores.append(f1_score(y_tr, y_tr_pred, average="macro", zero_division=0))
        val_f1_macro_scores.append(f1_score(y_val, y_val_pred, average="macro", zero_division=0))

        # Store predictions
        all_val_preds.extend(y_val_pred)
        all_val_true.extend(y_val)
        all_train_preds.extend(y_tr_pred)
        all_train_true.extend(y_tr)

    train_mean_acc = np.mean(train_scores)
    val_mean_acc = np.mean(val_scores)

    train_f1_mean = np.mean(train_f1_scores)
    train_f1_macro_mean = np.mean(train_f1_macro_scores)
    train_precision_mean = np.mean(train_precision_scores)
    train_recall_mean = np.mean(train_recall_scores)

    val_f1_mean = np.mean(val_f1_scores)
    val_f1_macro_mean = np.mean(val_f1_macro_scores)
    val_precision_mean = np.mean(val_precision_scores)
    val_recall_mean = np.mean(val_recall_scores)

    # === STDs (validation) ===
    val_std_acc = np.std(val_scores)
    val_std_f1 = np.std(val_f1_scores)
    val_std_f1_macro = np.std(val_f1_macro_scores)
    val_std_precision = np.std(val_precision_scores)
    val_std_recall = np.std(val_recall_scores)

    return (
        (train_mean_acc,
         train_f1_mean,
         train_f1_macro_mean,
         train_precision_mean,
         train_recall_mean),

        (val_mean_acc,
         val_f1_mean,
         val_f1_macro_mean,
         val_precision_mean,
         val_recall_mean),

        (val_std_acc,
         val_std_f1,
         val_std_f1_macro,
         val_std_precision,
         val_std_recall)
    )

In [17]:
def objective_random_forest(trial, x_train_cv, y_binary):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_features": trial.suggest_float("max_features", 0.1, 0.6),
        "min_samples_split": trial.suggest_int("min_samples_split", 5, 30),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 10),
        "random_state": seed.get_seed(),
        "class_weight": "balanced_subsample"
    }
    model = RandomForestClassifier(**params)

    with mlflow.start_run(nested=True, run_name=f"{class_name}_RF_trial_{trial.number}"):
        (train_means, val_means, val_stds) = cross_val_score_with_logging(
            model, x_train_cv, y_binary, trial.number,
            class_name, is_xgboost=False)

        (train_acc, train_f1_binary,
         train_f1_macro, train_precision,
         train_recall) = train_means

        (val_acc, val_f1_binary, val_f1_macro,
         val_precision, val_recall) = val_means

        (val_std_acc, val_std_f1_binary, val_std_f1_macro,
         val_std_precision, val_std_recall) = val_stds

        mlflow.set_tag("model_type", "RandomForest")
        mlflow.set_tag("kfold_splits", 5)
        mlflow.log_params(params)

        mlflow.log_metrics({
            "train_accuracy": train_acc,
            "train_f1_binary": train_f1_binary,
            "train_f1_macro": train_f1_macro,
            "train_precision": train_precision,
            "train_recall": train_recall,

            "val_accuracy": val_acc,
            "val_f1_binary": val_f1_binary,
            "val_f1_macro": val_f1_macro,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "val_std_accuracy": val_std_acc,
            "val_std_f1_binary": val_std_f1_binary,
            "val_std_f1_macro": val_std_f1_macro,
            "val_std_precision": val_std_precision,
            "val_std_recall": val_std_recall,
        })

        full_metrics = {
            "train": {
                "accuracy": train_acc,
                "f1_binary": train_f1_binary,
                "f1_macro": train_f1_macro,
                "precision": train_precision,
                "recall": train_recall,
            },
            "validation": {
                "accuracy_mean": val_acc,
                "accuracy_std": val_std_acc,
                "f1_binary_mean": val_f1_binary,
                "f1_binary_std": val_std_f1_binary,
                "f1_macro_mean": val_f1_macro,
                "f1_macro_std": val_std_f1_macro,
                "precision_mean": val_precision,
                "precision_std": val_std_precision,
                "recall_mean": val_recall,
                "recall_std": val_std_recall,
            },
            "params": params,
            "trial_number": trial.number,
            "class_name": class_name
        }

        mlflow.log_dict(full_metrics, "full_metrics.json")

    return val_f1_binary

In [18]:
def objective_xgboost(trial, x_train_cv, y_binary):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 150, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "gamma": trial.suggest_float("gamma", 0.3, 5.0),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "use_label_encoder": False,
        "eval_metric": "logloss",
        "random_state": seed.get_seed()
    }
    model = XGBClassifier(**params)
    with mlflow.start_run(nested=True, run_name=f"{class_name}_XGB_trial_{trial.number}"):
        (train_means, val_means, val_stds) = cross_val_score_with_logging(
            model, x_train_cv, y_binary, trial.number,
            class_name, is_xgboost=True)
        
        (train_acc, train_f1_binary,
         train_f1_macro, train_precision,
         train_recall) = train_means

        (val_acc, val_f1_binary, val_f1_macro,
         val_precision, val_recall) = val_means

        (val_std_acc, val_std_f1_binary, val_std_f1_macro,
         val_std_precision, val_std_recall) = val_stds
                                                                  
        mlflow.set_tag("model_type", "XGBoost")
        mlflow.set_tag("kfold_splits", 5)
        mlflow.log_params(params)
        mlflow.log_metrics({
            "train_accuracy": train_acc,
            "train_f1_binary": train_f1_binary,
            "train_f1_macro": train_f1_macro,
            "train_precision": train_precision,
            "train_recall": train_recall,

            "val_accuracy": val_acc,
            "val_f1_binary": val_f1_binary,
            "val_f1_macro": val_f1_macro,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "val_std_accuracy": val_std_acc,
            "val_std_f1_binary": val_std_f1_binary,
            "val_std_f1_macro": val_std_f1_macro,
            "val_std_precision": val_std_precision,
            "val_std_recall": val_std_recall,
        })

        full_metrics = {
            "train": {
                "accuracy": train_acc,
                "f1_binary": train_f1_binary,
                "f1_macro": train_f1_macro,
                "precision": train_precision,
                "recall": train_recall,
            },
            "validation": {
                "accuracy_mean": val_acc,
                "accuracy_std": val_std_acc,
                "f1_binary_mean": val_f1_binary,
                "f1_binary_std": val_std_f1_binary,
                "f1_macro_mean": val_f1_macro,
                "f1_macro_std": val_std_f1_macro,
                "precision_mean": val_precision,
                "precision_std": val_std_precision,
                "recall_mean": val_recall,
                "recall_std": val_std_recall,
            },
            "params": params,
            "trial_number": trial.number,
            "class_name": class_name
        }

        mlflow.log_dict(full_metrics, "full_metrics.json")

    return val_f1_binary


In [22]:
for class_name in label_encoder.classes_:
    class_id = label_encoder.transform([class_name])[0]
    y_binary = (y_train == class_id).astype(int)

    print(f"\n=== Optuna tuning for class '{class_name}' (ID={class_id}) ===")
    def objective_xgb_wrapped(trial):
        return objective_xgboost(trial, x_train_cv=x_train, y_binary=y_binary)

    mlflow.set_experiment(f"Tuning_XGB_{class_name}_img_descriptor_new_features_fixed_2")
    study_xgb = optuna.create_study(direction="maximize")
    study_xgb.optimize(objective_xgb_wrapped, n_trials=100)

    def objective_rf_wrapped(trial):
        return objective_random_forest(trial, x_train_cv=x_train, y_binary=y_binary)

    mlflow.set_experiment(f"Tuning_RF_{class_name}_img_descriptor_new_features_fixed_2")
    study_rf = optuna.create_study(direction="maximize")
    study_rf.optimize(objective_rf_wrapped, n_trials=100)


=== Optuna tuning for class 'Blob' (ID=0) ===


[I 2026-04-08 14:57:21,290] A new study created in memory with name: no-name-25b4137e-6b1b-48e2-89f5-d60e5f11ca5c
[I 2026-04-08 14:57:36,491] Trial 0 finished with value: 0.8887460920242377 and parameters: {'n_estimators': 460, 'learning_rate': 0.22637025092259258, 'gamma': 0.9664498332358293, 'max_depth': 11}. Best is trial 0 with value: 0.8887460920242377.
[I 2026-04-08 14:57:40,922] Trial 1 finished with value: 0.884092443904089 and parameters: {'n_estimators': 182, 'learning_rate': 0.21217565782262518, 'gamma': 0.6889750334025491, 'max_depth': 4}. Best is trial 0 with value: 0.8887460920242377.
[I 2026-04-08 14:57:55,894] Trial 2 finished with value: 0.8769386444383592 and parameters: {'n_estimators': 357, 'learning_rate': 0.2380874265125386, 'gamma': 3.865101564941872, 'max_depth': 12}. Best is trial 0 with value: 0.8887460920242377.
[I 2026-04-08 14:58:11,564] Trial 3 finished with value: 0.8752765310212117 and parameters: {'n_estimators': 454, 'learning_rate': 0.0504747306266666


=== Optuna tuning for class 'Diffusion Hit' (ID=1) ===


[I 2026-04-08 16:08:03,763] A new study created in memory with name: no-name-5036b212-eb94-48f9-b65a-574bf2866be6
[I 2026-04-08 16:08:09,463] Trial 0 finished with value: 0.9389473684210528 and parameters: {'n_estimators': 443, 'learning_rate': 0.21225573258606018, 'gamma': 0.9336248781451146, 'max_depth': 9}. Best is trial 0 with value: 0.9389473684210528.
[I 2026-04-08 16:08:17,296] Trial 1 finished with value: 0.9671826625386999 and parameters: {'n_estimators': 452, 'learning_rate': 0.2695262739740319, 'gamma': 4.2094072516152155, 'max_depth': 4}. Best is trial 1 with value: 0.9671826625386999.
[I 2026-04-08 16:08:23,969] Trial 2 finished with value: 0.9389473684210528 and parameters: {'n_estimators': 484, 'learning_rate': 0.2757770480218174, 'gamma': 1.6610475835421832, 'max_depth': 10}. Best is trial 1 with value: 0.9671826625386999.
[I 2026-04-08 16:08:28,628] Trial 3 finished with value: 0.9539473684210528 and parameters: {'n_estimators': 260, 'learning_rate': 0.1609248065181341


=== Optuna tuning for class 'Electron' (ID=2) ===


[I 2026-04-08 16:54:19,864] A new study created in memory with name: no-name-b25f11d0-eb1d-4fa0-894b-1080f8db5a8a
[I 2026-04-08 16:54:54,035] Trial 0 finished with value: 0.5965674580812033 and parameters: {'n_estimators': 368, 'learning_rate': 0.18118951079682516, 'gamma': 4.887620639529779, 'max_depth': 8}. Best is trial 0 with value: 0.5965674580812033.
[I 2026-04-08 16:55:08,055] Trial 1 finished with value: 0.6030215817052997 and parameters: {'n_estimators': 310, 'learning_rate': 0.0961578119550938, 'gamma': 0.5390370749652018, 'max_depth': 4}. Best is trial 1 with value: 0.6030215817052997.
[I 2026-04-08 16:55:25,568] Trial 2 finished with value: 0.5944366595373307 and parameters: {'n_estimators': 423, 'learning_rate': 0.2519847754849235, 'gamma': 1.137210568660449, 'max_depth': 4}. Best is trial 1 with value: 0.6030215817052997.
[I 2026-04-08 16:55:58,329] Trial 3 finished with value: 0.6065211548912005 and parameters: {'n_estimators': 287, 'learning_rate': 0.17284721709536346, 


=== Optuna tuning for class 'Muon' (ID=3) ===


2026/04/08 19:56:01 INFO mlflow.tracking.fluent: Experiment with name 'Tuning_XGB_Muon_img_descriptor_new_features_fixed_2' does not exist. Creating a new experiment.
[I 2026-04-08 19:56:01,986] A new study created in memory with name: no-name-e3c3f109-3a1b-492a-ac6d-7a30bf5ce673
[I 2026-04-08 19:56:28,264] Trial 0 finished with value: 0.9466712500287008 and parameters: {'n_estimators': 430, 'learning_rate': 0.07480519143214066, 'gamma': 4.884704983728758, 'max_depth': 12}. Best is trial 0 with value: 0.9466712500287008.
[I 2026-04-08 19:56:33,625] Trial 1 finished with value: 0.9492668517185165 and parameters: {'n_estimators': 188, 'learning_rate': 0.05573355213654621, 'gamma': 3.8668575037699964, 'max_depth': 5}. Best is trial 1 with value: 0.9492668517185165.
[I 2026-04-08 19:56:41,355] Trial 2 finished with value: 0.9547287205855731 and parameters: {'n_estimators': 223, 'learning_rate': 0.03191213033229964, 'gamma': 1.3759132821360835, 'max_depth': 6}. Best is trial 2 with value: 0


=== Optuna tuning for class 'Others' (ID=4) ===


2026/04/08 23:16:06 INFO mlflow.tracking.fluent: Experiment with name 'Tuning_XGB_Others_img_descriptor_new_features_fixed_2' does not exist. Creating a new experiment.
[I 2026-04-08 23:16:06,468] A new study created in memory with name: no-name-096108a3-2ef1-49ad-a6bf-850e94714a61
[I 2026-04-08 23:16:14,102] Trial 0 finished with value: 0.6473957433440666 and parameters: {'n_estimators': 239, 'learning_rate': 0.10424618479548864, 'gamma': 4.895089136002514, 'max_depth': 5}. Best is trial 0 with value: 0.6473957433440666.
[I 2026-04-08 23:16:24,760] Trial 1 finished with value: 0.6498688292001062 and parameters: {'n_estimators': 201, 'learning_rate': 0.2522706919949694, 'gamma': 1.70745424079626, 'max_depth': 11}. Best is trial 1 with value: 0.6498688292001062.
[I 2026-04-08 23:16:37,595] Trial 2 finished with value: 0.6499331906717627 and parameters: {'n_estimators': 343, 'learning_rate': 0.17518998981295097, 'gamma': 3.002197202248642, 'max_depth': 7}. Best is trial 2 with value: 0.6